# 00 · The question and the assets

**Goal:** distinguish a potentially useful teaching mechanism from
ordinary gains caused by additional training.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](../../../notebooks/synthetic_training/README.md)
· [HAIC setup and launch commands](../../../slurm/synthetic-training/README.md)

In [1]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

Run: /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/outputs/synthetic-training/pilot-01
Context representation: vjepa; device: cuda


## 1. Understand the experiment in one example

A pose estimator predicts twelve visible body landmarks. A **lesson**
is a small labeled set of rendered AMASS images, such as oblique views
or partly occluded people. A **probe** is a common short training update.
A **selector** uses the estimator's response to choose its next lesson.

We need four findings in sequence:

1. Synthetic lessons can improve real accuracy beyond equal-budget replay.
2. Different students or settings benefit from different lessons.
3. Target-video prediction changes select better lessons than current
   weaknesses and source learning progress alone.
4. A selector learned on source trials transfers to a held architecture.

Improved confidence or lower training loss does not establish real
accuracy. A useful nearest-neighbor selector is enough to test the
mechanism; a more complicated teacher is not itself the contribution.

In [2]:
display(pd.DataFrame([
    ("train", "Fit source outcome predictors", "AMASS source references"),
    ("validation", "Choose selector settings and comparators", "Separate source students and references"),
    ("held", "Test transfer to an unseen architecture", "Never enters source fitting or model selection"),
], columns=["Student role", "Purpose", "Reference boundary"]))
display(pd.DataFrame(cfg.students))

,Student role,Purpose,Reference boundary
0,train,Fit source outcome predictors,AMASS source references
1,validation,Choose selector settings and comparators,Separate source students and references
2,held,Test transfer to an unseen architecture,Never enters source fitting or model selection


,student_id,family,role,config,checkpoint
0,rtmpose_m,rtmpose,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
1,hrnet_w32,hrnet,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
2,rtmpose_s,rtmpose,validation,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
3,hrnet_w48,hrnet,validation,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
4,vitpose_base,vitpose,held,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...


## 2. Inspect actual asset availability

This reads the repository's AMASS and GAVD manifests and checks local
paths. It does not infer availability from a manifest count or claim
that a checkpoint has successfully loaded. The following stage needs
full-body AMASS files, licensed body models, compatible texture/UV
assets, photographic backgrounds, and labeled COCO replay data.

Model configuration files and their corresponding released weights
must be present. Use the HAIC guide's isolated MMPose environment when
the main project's PyTorch version is incompatible with MMCV.

In [3]:
started = perf_counter()
inventory = workflow.inventory(cfg)
show_result(inventory)
print(f"Inventory took {perf_counter() - started:.1f} seconds.")

### amass

,relative_path,source_dataset,parent_path,subject_id_candidate,motion_id,sha256,npz_keys,num_frames,pose_width,trans_frames,...,mocap_framerate,gender,status,error,person_id,original_split,role,raw_path,available,duration_s
0,BioMotionLab_NTroje/rub044/0000_treadmill_norm...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0000_treadmill_norm,a1629a596af402d418585c7aec0d1ae57c4c8c5fd3c89a...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",3182,156,3182,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,26.508333
1,BioMotionLab_NTroje/rub044/0001_treadmill_fast...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0001_treadmill_fast,e8576e8799b2d5be54d752df71ead342ee87ddc6aa3d92...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",2833,156,2833,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,23.600000
2,BioMotionLab_NTroje/rub044/0002_treadmill_slow...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0002_treadmill_slow,7b92530ae5aec450baa075bf41da50a406e70b7b51b9cf...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",2792,156,2792,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,23.258333
3,BioMotionLab_NTroje/rub044/0003_treadmill_jog_...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0003_treadmill_jog,6950e2f140d822d86f92a8b12f9fc7ab16366afd465fcd...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",2692,156,2692,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,22.425000
4,BioMotionLab_NTroje/rub044/0004_motorcycle_pos...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0004_motorcycle,bfde7f928876d1b466627fda0996681a6309a70218fdfa...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",370,156,370,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,3.075000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8849,MPI_HDM05/tr/HDM_tr_05-03_02_120_poses.npz,MPI_HDM05,MPI_HDM05/tr,MPI_HDM05::tr,HDM_tr_05-03_02_120,6c5d42b7513e63b743adda9e3741fd771c919596ee24ea...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",3634,156,3634,...,120.0,male,ok,NaN,MPI_HDM05::tr,train,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,30.275000
8850,MPI_HDM05/tr/HDM_tr_05-03_03_120_poses.npz,MPI_HDM05,MPI_HDM05/tr,MPI_HDM05::tr,HDM_tr_05-03_03_120,cf048420fff08d7511edbf60689c8d4bdfcebfa199f5d9...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",3338,156,3338,...,120.0,male,ok,NaN,MPI_HDM05::tr,train,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,27.808333
8851,MPI_HDM05/tr/HDM_tr_05-03_04_120_poses.npz,MPI_HDM05,MPI_HDM05/tr,MPI_HDM05::tr,HDM_tr_05-03_04_120,2abcc2c8ecfe9e29da9c8cd27c589c2902e5fafe774c4c...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",6453,156,6453,...,120.0,male,ok,NaN,MPI_HDM05::tr,train,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,53.766667
8852,MPI_HDM05/tr/HDM_tr_06-01_01_120_poses.npz,MPI_HDM05,MPI_HDM05/tr,MPI_HDM05::tr,HDM_tr_06-01_01_120,46c7e02910f19e4bc0bca7214e53284de667c52e64dd60...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",10768,156,10768,...,120.0,male,ok,NaN,MPI_HDM05::tr,train,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,89.725000


### gavd

,sequence_id,video_id,url,first_frame,last_frame,n_annotated_frames,source_height,dataset_annotation,gait_pattern_annotation,cam_view,video_path,available,group_id,group_unit,role
0,cljan9b4p00043n6ligceanyp,B5hrxKe2nP8,https://www.youtube.com/watch?v=B5hrxKe2nP8,1757,2268,512,720,Abnormal Gait,parkinsons,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,B5hrxKe2nP8,recording_identity_not_known,external_observational
1,cljanb45y00083n6lmh1qhydd,B5hrxKe2nP8,https://www.youtube.com/watch?v=B5hrxKe2nP8,2532,2746,215,720,Abnormal Gait,parkinsons,left side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,B5hrxKe2nP8,recording_identity_not_known,external_observational
2,cljao8kyf000d3n6l0x9kgmav,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,1,148,148,720,Abnormal Gait,abnormal,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
3,cljaoak47000i3n6lsrb9rit9,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,205,355,151,720,Abnormal Gait,abnormal,left side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
4,cljaob36l000m3n6l9xokjqww,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,382,813,432,720,Abnormal Gait,abnormal,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1869,cllepgmuf001f3o6lms5hhzg1,FFki8FtaByw,https://www.youtube.com/watch?v=FFki8FtaByw,8156,8277,122,480,Abnormal Gait,abnormal,left side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,FFki8FtaByw,recording_identity_not_known,external_observational
1870,cllephqq9001m3o6li43x1ynu,FFki8FtaByw,https://www.youtube.com/watch?v=FFki8FtaByw,8309,8374,66,480,Abnormal Gait,abnormal,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,FFki8FtaByw,recording_identity_not_known,external_observational
1871,cllepj4kb001s3o6le0fhjor3,FFki8FtaByw,https://www.youtube.com/watch?v=FFki8FtaByw,11118,11289,172,480,Abnormal Gait,abnormal,left side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,FFki8FtaByw,recording_identity_not_known,external_observational
1872,cllfyfckd00043o6liq5zlwv2,Ha9LKXZfWBQ,https://www.youtube.com/watch?v=Ha9LKXZfWBQ,1502,1620,119,480,Abnormal Gait,myopathic,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,Ha9LKXZfWBQ,recording_identity_not_known,external_observational


### assets

,asset,path,exists
0,amass_root,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True
1,body_model_root,/hai/scratch/tedmui/body_models,True
2,render_texture_dir,,False
3,render_background_dir,,False
4,render_uv_path,,False
5,coco_image_root,/hai/scratch/tedmui/coco/train2017,True
6,coco_annotations_json,/hai/scratch/tedmui/coco/annotations/person_ke...,True
7,gavd_video_root,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True
8,gavd_reservation_csv,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True
9,context_repo,/hai/scratch/tedmui/vendor/vjepa2,True


### students

,student_id,family,role,config,checkpoint
0,rtmpose_m,rtmpose,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
1,hrnet_w32,hrnet,train,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
2,rtmpose_s,rtmpose,validation,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
3,hrnet_w48,hrnet,validation,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...
4,vitpose_base,vitpose,held,/hai/scratch/tedmui/alexpose/experiments/sjepa...,/hai/scratch/tedmui/alexpose/experiments/sjepa...


Inventory took 11.8 seconds.


## 3. Keep the interpretation narrow

GAVD supplies real video, not existing accurate twelve-joint reference
coordinates. Those visible landmarks will be independently annotated.
AMASS projected labels describe the rendered geometry. Neither source
establishes forces, 3D clinical accuracy, or hidden-joint accuracy.

A simple-context run can test the teaching pipeline, but cannot support
a JEPA-specific claim. A JEPA run must actually load the configured
encoder and compare its features with simpler context representations.

**Continue when:** assets are available and you can prepare useful RGB
lessons and independent references. Missing appearance or human labels
is a real dependency, not a reason to substitute model predictions.

Next: [01 · Prepare source data](../../../notebooks/synthetic_training/01_prepare_source_data.ipynb).